# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [ ]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


This creates a dataset of 400 orders with a fixed random seed, assigning each order a vendor, category, quantity, and price. After that, it previews the first five rows.

### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [ ]:
# TODO
df['revenue'] = df['qty'] * df['price']
total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()
print(f"Total revenue: ${total_revenue:,.2f}")
print(f"Total units sold: {total_units}")

Total revenue: $8,520.00
Total units sold: 783


This computes revenue per order (qty × price), then sums it to get total revenue ($8,520.00) and total units sold (783) across all 400 orders.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [ ]:
# TODO
by_category = (
    df.groupby('category')['revenue']
    .sum()
    .sort_values(ascending=False)
    .to_frame()
)
by_category['pct_of_total'] = (by_category['revenue'] / total_revenue * 100).round(1)
by_category

,revenue,pct_of_total
category,,
Food,4293.0,50.4
Merch,1771.5,20.8
Drink,1554.0,18.2
RainGear,901.5,10.6


This groups orders by category, sums revenue for each, sorts from highest to lowest, and adds a pct_of_total column so you can see each category's share alongside its raw dollar total. Food has the highest revenue at $4,293.00 (50.4%), followed by Merch, Drink, and RainGear.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [ ]:
# TODO
by_vendor = (
    df.groupby('vendor_id')['revenue']
    .agg(avg_order_revenue='mean', order_count='count')
    .sort_values('avg_order_revenue', ascending=False)
)
by_vendor

,avg_order_revenue,order_count
vendor_id,,
V-01,22.595745,94
V-18,21.750000,108
V-05,20.580645,93
V-10,20.314286,105


This groups the orders by vendor, then calculates each vendor's average order revenue along with how many orders they had. When sorting by average revenue, it puts the top performers first. However, showing the order count next to it matters because a vendor with a good average from only a few orders isn't necessarily doing better than one with a slightly lower average across many orders.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [ ]:
# TODO
merch_share = by_category.loc['Merch', 'pct_of_total']
print(f"Merch share of revenue: {merch_share}%")

Merch share of revenue: 20.8%


This gets Merch's pct_of_total value out of the by_category table computed in Q2 and prints it, confirming Merch makes up 20.8% of total revenue.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [ ]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor

joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

unmatched_ids = joined.loc[joined['vendor_name'].isna(), 'vendor_id'].unique()
print("Unmatched vendor id(s):", list(unmatched_ids))

print("Row count unchanged:", len(joined) == len(df))
print("Revenue total unchanged:", abs(joined['revenue'].sum() - df['revenue'].sum()) < 0.01)

joined['vendor_name'] = joined['vendor_name'].fillna(joined['vendor_id'])
joined.head()

Unmatched vendor id(s): ['V-18']
Row count unchanged: True
Revenue total unchanged: True


,vendor_id,category,qty,price,revenue,vendor_name
0,V-10,Drink,2,24.0,48.0,Cav Merch North
1,V-18,RainGear,1,12.0,12.0,V-18
2,V-18,Drink,3,4.5,13.5,V-18
3,V-10,Food,2,12.0,24.0,Cav Merch North
4,V-18,Drink,3,7.5,22.5,V-18


The left-merges vendor display names onto the orders, flags any vendor_id that had no match in the lookup table (V-18), confirms row count and revenue total are unchanged after the merge, and then fills the missing name with the raw id as a fallback so every row still has a readable label.

**The unmatched vendor, and what I did about it:** The vendor id V-18 doesn't appear in the vendor_names lookup table, so after the left join, its vendor_name came back as NaN for every one of its 108 orders. Rather than dropping those rows (which would have changed both the row count and the revenue total), I filled the missing name with the vendor's own id (fillna(joined['vendor_id'])), so V-18 shows up labeled as "V-18" in the pivot table instead of breaking the merge.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [ ]:
# TODO
pd.pivot_table(
    joined,
    values='revenue',
    index='vendor_name',
    columns='category',
    aggfunc='sum',
    margins=True,
    margins_name='Total',
)

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
V-18,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


This builds a pivot table with vendors as rows and categories as columns, summing revenue in each cell, and adds row/column totals (margins=True).

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [ ]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a) Food dominates revenue (4,293 dollars, 50.4% of 8,520 dollars total), while RainGear falls behind at just 901.50 dollars (10.6%). V-01 has the highest average order value ($22.60), but V-18 brings in the most total revenue at 2,349 dollars simply by running more orders (108). The two takeaways from the next game is double down on Food since demand is there, and get V-18 registered in the vendor lookup so it shows up in reporting.

b) The Q3 vendor average comparison is the least trustworthy. Sample sizes are fine (93–108 orders each), but the averages sit within about 2 dollars within each other ($20.31–$22.60). This is close enough that the ranking could flip with different random price draws. On this alone, I would not call V-01 the best vendor.